# nn-module-subclass composite — cx20: subclass nn.Module and wrap a raw tensor as nn.Parameter

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `nn-module-subclass`, `nn-parameter-wrap`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "nn-module-subclass"
DD_ATOM_IDS = ["nn-module-subclass", "nn-parameter-wrap"]
DD_SUBTOPICS = ["PyTorch: nn.Module subclassing", "PyTorch: nn.Parameter"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

When ARENA asks you to roll your own `Linear` / `Embedding` / `BatchNorm`, you don't get to lean on `nn.Linear` — you have to manage the **raw weight tensor** yourself. The combination:
- **nn-module-subclass** — the usual `class Foo(nn.Module)` + `super().__init__()` + `forward` scaffold.
- **nn-parameter-wrap** — wrap the raw tensor with `nn.Parameter(tensor)`. The bare wrapper marks the tensor as a *learnable* parameter; assigning it to an attribute (`self.weight = nn.Parameter(...)`) registers it in `self._parameters`, which is what makes it show up in `.parameters()` and get moved by `.to(device)`.

**Why not just `self.weight = t.randn(...)`?** A raw tensor assigned to a module attribute is invisible to `.parameters()` (it lives in `__dict__`, not `_parameters`). The optimizer would silently never update it.

**Anatomy.**
1. `super().__init__()` first.
2. `w = t.empty(out_dim, in_dim); nn.init.kaiming_uniform_(w, a=5**0.5)` — Kaiming-like init matching `nn.Linear`'s default scheme.
3. `self.weight = nn.Parameter(w)` — the wrap + assign that registers the tensor.
4. `forward(self, x)`: `return x @ self.weight.T` (no bias in this drill).

### Composite Exercise — subclass nn.Module and wrap a raw tensor as nn.Parameter

**Atoms exercised together**: `nn-module-subclass`, `nn-parameter-wrap`

Define a class `MyLinear(nn.Module)` and a builder `cx20_build_my_linear(in_dim, out_dim)` that returns an instance.

`MyLinear.__init__(self, in_dim, out_dim)` must:
1. Call `super().__init__()`.
2. Create a `(out_dim, in_dim)` weight tensor (use `t.empty` + `nn.init.kaiming_uniform_(w, a=5**0.5)` so init matches `nn.Linear`).
3. **Wrap** it as `nn.Parameter(w)` and assign to `self.weight`. Do NOT assign the raw tensor — the test verifies the type.

`MyLinear.forward(self, x)` returns `x @ self.weight.T` (no bias here — cx22 does the full affine).

The test checks: type is `nn.Parameter`, `requires_grad=True`, registered in `_parameters` (NOT in `__dict__`), and that `.parameters()` yields exactly one tensor.

In [ ]:
class MyLinear(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # Allocate the raw weight tensor and Kaiming-init it (same scheme nn.Linear uses).
        w = t.empty(out_dim, in_dim)
        nn.init.kaiming_uniform_(w, a=5 ** 0.5)
        # Atom (nn-parameter-wrap): wrap the raw tensor so it gets registered in
        # self._parameters by nn.Module.__setattr__. requires_grad=True by default.
        self.weight = nn.Parameter(w)

    def forward(self, x):
        # (B, in_dim) @ (in_dim, out_dim) = (B, out_dim)
        return x @ self.weight.T


def cx20_build_my_linear(in_dim: int, out_dim: int) -> 'MyLinear':
    return MyLinear(in_dim, out_dim)


<details><summary>Show solution — cx20</summary>

```python
class MyLinear(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # Allocate the raw weight tensor and Kaiming-init it (same scheme nn.Linear uses).
        w = t.empty(out_dim, in_dim)
        nn.init.kaiming_uniform_(w, a=5 ** 0.5)
        # Atom (nn-parameter-wrap): wrap the raw tensor so it gets registered in
        # self._parameters by nn.Module.__setattr__. requires_grad=True by default.
        self.weight = nn.Parameter(w)

    def forward(self, x):
        # (B, in_dim) @ (in_dim, out_dim) = (B, out_dim)
        return x @ self.weight.T


def cx20_build_my_linear(in_dim: int, out_dim: int) -> 'MyLinear':
    return MyLinear(in_dim, out_dim)
```

`nn.Parameter` is a subclass of `Tensor` that `nn.Module.__setattr__` specifically watches for — it slots into `_parameters` not `_modules`. The wrap is the contract. Without it, the tensor is a stray attribute and the optimizer never sees it.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx20'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx20',
        'subtopics': ["PyTorch: nn.Module subclassing", "PyTorch: nn.Parameter"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()